## _import libraries_

In [58]:
# !pip install "numpy<2.4" --force-reinstall

   ---------------------------------------- 0.0/12.8 MB ? eta -:--:--
   - -------------------------------------- 0.5/12.8 MB 3.8 MB/s eta 0:00:04
   ---- ----------------------------------- 1.3/12.8 MB 4.0 MB/s eta 0:00:03
   ---- ----------------------------------- 1.3/12.8 MB 4.0 MB/s eta 0:00:03
   -------- ------------------------------- 2.6/12.8 MB 3.5 MB/s eta 0:00:03
   --------- ------------------------------ 2.9/12.8 MB 3.6 MB/s eta 0:00:03
   --------- ------------------------------ 3.1/12.8 MB 2.9 MB/s eta 0:00:04
   -------------- ------------------------- 4.7/12.8 MB 3.4 MB/s eta 0:00:03
   ------------------ --------------------- 5.8/12.8 MB 3.7 MB/s eta 0:00:02
   ------------------- -------------------- 6.3/12.8 MB 3.6 MB/s eta 0:00:02
   --------------------- ------------------ 6.8/12.8 MB 3.5 MB/s eta 0:00:02
   ----------------------- ---------------- 7.6/12.8 MB 3.5 MB/s eta 0:00:02
   -------------------------- ------------- 8.4/12.8 MB 3.5 MB/s eta 0:00:02
   ---

In [63]:
# import sys
# !{sys.executable} -m pip install scikit-learn

  Using cached threadpoolctl-3.6.0-py3-none-any.whl.metadata (13 kB)
   ---------------------------------------- 0.0/8.9 MB ? eta -:--:--
   ---------------------------------------- 0.0/8.9 MB ? eta -:--:--
   -- ------------------------------------- 0.5/8.9 MB 2.8 MB/s eta 0:00:03
   ----- ---------------------------------- 1.3/8.9 MB 3.5 MB/s eta 0:00:03
   --------- ------------------------------ 2.1/8.9 MB 3.7 MB/s eta 0:00:02
   ----------- ---------------------------- 2.6/8.9 MB 3.6 MB/s eta 0:00:02
   --------------- ------------------------ 3.4/8.9 MB 3.5 MB/s eta 0:00:02
   ------------------ --------------------- 4.2/8.9 MB 3.5 MB/s eta 0:00:02
   ---------------------- ----------------- 5.0/8.9 MB 3.6 MB/s eta 0:00:02
   ------------------------- -------------- 5.8/8.9 MB 3.6 MB/s eta 0:00:01
   ----------------------------- ---------- 6.6/8.9 MB 3.7 MB/s eta 0:00:01
   --------------------------------- ------ 7.3/8.9 MB 3.7 MB/s eta 0:00:01
   ------------------------------

In [1]:
import pandas as pd
import numpy as np
import re
from surprise import Dataset, Reader, SVD, accuracy
from surprise.model_selection import train_test_split
from sklearn.metrics import root_mean_squared_error, mean_absolute_error
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

## _load Dataset_

In [2]:
movies  = pd.read_csv("movies.csv")
ratings = pd.read_csv("ratings.csv")
movies.head(10)


,movieId,title,genres
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
1,2,Jumanji (1995),Adventure|Children|Fantasy
2,3,Grumpier Old Men (1995),Comedy|Romance
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance
4,5,Father of the Bride Part II (1995),Comedy
5,6,Heat (1995),Action|Crime|Thriller
6,7,Sabrina (1995),Comedy|Romance
7,8,Tom and Huck (1995),Adventure|Children
8,9,Sudden Death (1995),Action
9,10,GoldenEye (1995),Action|Adventure|Thriller


In [3]:
ratings.head(10)

,userId,movieId,rating,timestamp
0,1,1,4.0,964982703
1,1,3,4.0,964981247
2,1,6,4.0,964982224
3,1,47,5.0,964983815
4,1,50,5.0,964982931
5,1,70,3.0,964982400
6,1,101,5.0,964980868
7,1,110,4.0,964982176
8,1,151,5.0,964984041
9,1,157,5.0,964984100


In [4]:
print(movies.info())
print(ratings.info())

<class 'pandas.DataFrame'>
RangeIndex: 9742 entries, 0 to 9741
Data columns (total 3 columns):
 #   Column   Non-Null Count  Dtype
---  ------   --------------  -----
 0   movieId  9742 non-null   int64
 1   title    9742 non-null   str  
 2   genres   9742 non-null   str  
dtypes: int64(1), str(2)
memory usage: 626.1 KB
None
<class 'pandas.DataFrame'>
RangeIndex: 100836 entries, 0 to 100835
Data columns (total 4 columns):
 #   Column     Non-Null Count   Dtype  
---  ------     --------------   -----  
 0   userId     100836 non-null  int64  
 1   movieId    100836 non-null  int64  
 2   rating     100836 non-null  float64
 3   timestamp  100836 non-null  int64  
dtypes: float64(1), int64(3)
memory usage: 3.1 MB
None


## _cleaning Data_

In [5]:
print(movies.isnull().sum())
print(ratings.isnull().sum())

movieId    0
title      0
genres     0
dtype: int64
userId       0
movieId      0
rating       0
timestamp    0
dtype: int64


In [6]:
print(movies.duplicated().sum())
print(ratings.duplicated().sum())


0
0


In [7]:
ratings.drop(columns=['timestamp'], inplace=True)

In [8]:
#  Clean Text (Genres)
def clean_genres(text):
    text = str(text).lower()
    text = text.replace('|', ' ')
    text = re.sub(r'[^a-z ]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

movies['genres'] = movies['genres'].apply(clean_genres)

In [9]:
#  Extract Year from Title
#def extract_year(title):
   # match = re.search(r'\((\d{4})\)', str(title))
    #if match:
        #return int(match.group(1))
    #return np.nan

#movies['year'] = movies['title'].apply(extract_year)

In [10]:
# Clean Title (Remove Year)

#def clean_title(title):
    #title = re.sub(r'\(\d{4}\)', '', str(title))  # remove year
    #return title.strip().lower()

#movies['clean_title'] = movies['title'].apply(clean_title)

In [11]:
 # Filter Rare Movies
#movie_rating_count = ratings.groupby('movieId')['rating'].count()

#popular_movies = movie_rating_count[movie_rating_count >= 20].index

#ratings = ratings[ratings['movieId'].isin(popular_movies)]
#movies = movies[movies['movieId'].isin(popular_movies)]

In [12]:
merged_df = pd.merge(ratings, movies, on="movieId")
merged_df.head(10)

,userId,movieId,rating,title,genres
0,1,1,4.0,Toy Story (1995),adventure animation children comedy fantasy
1,1,3,4.0,Grumpier Old Men (1995),comedy romance
2,1,6,4.0,Heat (1995),action crime thriller
3,1,47,5.0,Seven (a.k.a. Se7en) (1995),mystery thriller
4,1,50,5.0,"Usual Suspects, The (1995)",crime mystery thriller
5,1,70,3.0,From Dusk Till Dawn (1996),action comedy horror thriller
6,1,101,5.0,Bottle Rocket (1996),adventure comedy crime romance
7,1,110,4.0,Braveheart (1995),action drama war
8,1,151,5.0,Rob Roy (1995),action drama romance war
9,1,157,5.0,Canadian Bacon (1995),comedy war


## Collaborative Filtering

In [34]:
#prepeare data for surprise
# constrains
reader = Reader(rating_scale=(1, 5))
#user item matrix
data = Dataset.load_from_df(
    ratings[['userId', 'movieId', 'rating']],
    reader
)
print("Done")

Done


In [35]:
# split data 80% train - 20% test
trainset, testset = train_test_split(data, test_size=0.2, random_state=42)

In [36]:
# train SVD model
#model Hyperparameter Tuning
model=SVD(n_factors=50,n_epochs=20, random_state=42)
# in this step the model will make matrix factorization
## and extract Latent Factors that represent user tastes and movie features.
#  matrix factorization the model will devive the item matrix (big matrix) to smaller matrics
model.fit(trainset)

In [37]:
# predict and evaluate model
predictions=model.test(testset)

rmse =accuracy.rmse((predictions))
mae =accuracy.mae((predictions))

RMSE: 0.8775
MAE:  0.6742


In [38]:
def recommend_cf(user_id, n=10): 
    # all movies
    all_movie_ids = ratings['movieId'].unique()

    # movies user already watched
    watched =[]
    for index,row in ratings.iterrows():
        if row['userId']==user_id:
            watched.append(row['movieId'])

    # movies user haven't watched yet
    not_watched = []
    for m in all_movie_ids:
        if m not in watched:
            not_watched.append(m)

    # predict rating for each movie
    predictions_list = []
    for movie_id in not_watched:
        pred = model.predict(user_id, movie_id)
        predictions_list.append((movie_id, pred.est))

    # sort from highest to lowest
    pred_df = pd.DataFrame(predictions_list, columns=['movieId', 'score'])
    top_pred_df = pred_df.sort_values(by='score', ascending=False).head(n)

    result = []
    for index, row  in top_pred_df.iterrows():

       movie_info = movies[movies['movieId'] == row['movieId']]

       title = movie_info['title'].values[0]
       genres = movie_info['genres'].values[0]

       result.append({
        'title': title,
        'genres': genres,
        'predicted_rating': f"{row['score']:.2f}"
        })

    return pd.DataFrame(result)


In [39]:
# show top 10 movies for user 1
recommendations=recommend_cf(user_id=1,n=10)
print("top Movie Recommendations :")
print(recommendations)

top Movie Recommendations :
                                               title  \
0                   Shawshank Redemption, The (1994)   
1         Ghost in the Shell (Kôkaku kidôtai) (1995)   
2                         Kiss Kiss Bang Bang (2005)   
3                        Boondock Saints, The (2000)   
4                          Lawrence of Arabia (1962)   
5                             Rosemary's Baby (1968)   
6         Life Is Beautiful (La Vita è bella) (1997)   
7     Cinema Paradiso (Nuovo cinema Paradiso) (1989)   
8  Amelie (Fabuleux destin d'Amélie Poulain, Le) ...   
9                               Departed, The (2006)   

                          genres predicted_rating  
0                    crime drama             5.00  
1               animation sci fi             5.00  
2  comedy crime mystery thriller             5.00  
3    action crime drama thriller             5.00  
4            adventure drama war             5.00  
5          drama horror thriller           

# Content-Based Filtering

In [40]:
# Check Dataset
print(ratings.shape)
print(movies[['movieId', 'title', 'genres']].head())

(100836, 3)
   movieId                               title  \
0        1                    Toy Story (1995)   
1        2                      Jumanji (1995)   
2        3             Grumpier Old Men (1995)   
3        4            Waiting to Exhale (1995)   
4        5  Father of the Bride Part II (1995)   

                                        genres  
0  adventure animation children comedy fantasy  
1                   adventure children fantasy  
2                               comedy romance  
3                         comedy drama romance  
4                                       comedy  


In [41]:
# Remove Duplicate Movies
movies_content = movies[['movieId', 'title', 'genres']].drop_duplicates()
movies_content.reset_index(drop=True, inplace=True)
movies_content.head()

,movieId,title,genres
0,1,Toy Story (1995),adventure animation children comedy fantasy
1,2,Jumanji (1995),adventure children fantasy
2,3,Grumpier Old Men (1995),comedy romance
3,4,Waiting to Exhale (1995),comedy drama romance
4,5,Father of the Bride Part II (1995),comedy


In [42]:
# Handle Missing Values
movies_content['genres'] = movies_content['genres'].fillna('')

In [43]:
# Convert Genres to TF-IDF Matrix
tfidf = TfidfVectorizer(stop_words='english')
genre_matrix = tfidf.fit_transform(movies_content['genres'])
print(genre_matrix.shape)

(9742, 23)


In [44]:
#Calculate Cosine Similarity
cosine_sim = cosine_similarity(genre_matrix, genre_matrix)
print(cosine_sim.shape)

(9742, 9742)


In [45]:
# Create Movie Index Mapping
indices = pd.Series(movies_content.index,index=movies_content['title']).drop_duplicates()
indices.head()

title
Toy Story (1995)                      0
Jumanji (1995)                        1
Grumpier Old Men (1995)               2
Waiting to Exhale (1995)              3
Father of the Bride Part II (1995)    4
dtype: int64

In [46]:
# Recommendation Function
def recommend_cb(title, cosine_sim=cosine_sim):
    # Check if movie exists
    if title not in indices:
        return "Movie not found"
    # Get movie index
    idx = indices[title]
    # Get similarity scores
    sim_scores = list(enumerate(cosine_sim[idx]))
    # Sort movies
    sim_scores = sorted(sim_scores,key=lambda x: x[1],reverse=True)
    # Remove selected movie
    sim_scores = sim_scores[1:11]
    # Get movie indices
    movie_indices = [i[0] for i in sim_scores]
    # Return recommendations
    return movies_content[['title', 'genres']].iloc[movie_indices]

In [47]:
# Test Recommendation System
recommend_cb('Toy Story (1995)')

,title,genres
1706,Antz (1998),adventure animation children comedy fantasy
2355,Toy Story 2 (1999),adventure animation children comedy fantasy
2809,"Adventures of Rocky and Bullwinkle, The (2000)",adventure animation children comedy fantasy
3000,"Emperor's New Groove, The (2000)",adventure animation children comedy fantasy
3568,"Monsters, Inc. (2001)",adventure animation children comedy fantasy
6194,"Wild, The (2006)",adventure animation children comedy fantasy
6486,Shrek the Third (2007),adventure animation children comedy fantasy
6948,"Tale of Despereaux, The (2008)",adventure animation children comedy fantasy
7760,Asterix and the Vikings (Astérix et les Viking...,adventure animation children comedy fantasy
8219,Turbo (2013),adventure animation children comedy fantasy


In [48]:
# Test Another Movie
recommend_cb('Jumanji (1995)')

,title,genres
53,"Indian in the Cupboard, The (1995)",adventure children fantasy
109,"NeverEnding Story III, The (1994)",adventure children fantasy
767,Escape to Witch Mountain (1975),adventure children fantasy
1514,Darby O'Gill and the Little People (1959),adventure children fantasy
1556,Return to Oz (1985),adventure children fantasy
1617,"NeverEnding Story, The (1984)",adventure children fantasy
1618,"NeverEnding Story II: The Next Chapter, The (1...",adventure children fantasy
1799,Santa Claus: The Movie (1985),adventure children fantasy
3574,Harry Potter and the Sorcerer's Stone (a.k.a. ...,adventure children fantasy
6075,"Chronicles of Narnia: The Lion, the Witch and ...",adventure children fantasy


In [49]:
#Recommendation Function with Similarity Score
def recommend_with_score(title, cosine_sim=cosine_sim):

    if title not in indices:
        return "Movie not found"

    idx = indices[title]

    sim_scores = list(enumerate(cosine_sim[idx]))

    sim_scores = sorted(
        sim_scores,
        key=lambda x: x[1],
        reverse=True
    )

    sim_scores = sim_scores[1:11]

    movie_indices = [i[0] for i in sim_scores]

    scores = [i[1] for i in sim_scores]

    result = movies_content[['title', 'genres']].iloc[movie_indices]

    result['similarity_score'] = scores

    return result

In [50]:
# Test Function with Scores
recommend_with_score('Heat (1995)')

,title,genres,similarity_score
22,Assassins (1995),action crime thriller,1.0
138,Die Hard: With a Vengeance (1995),action crime thriller,1.0
156,"Net, The (1995)",action crime thriller,1.0
249,Natural Born Killers (1994),action crime thriller,1.0
417,Judgment Night (1993),action crime thriller,1.0
509,Batman (1989),action crime thriller,1.0
793,Die Hard (1988),action crime thriller,1.0
1306,Hard Rain (1998),action crime thriller,1.0
1315,"Replacement Killers, The (1998)",action crime thriller,1.0
1325,U.S. Marshals (1998),action crime thriller,1.0


## Hybrid Recommendation (CF + CB)

In [51]:
def recommend_hybrid(user_id, title, n=10, cf_weight=0.6, cb_weight=0.4):
    all_movies  = ratings['movieId'].unique()
    watched     = ratings[ratings['userId'] == user_id]['movieId'].values
    not_watched = [m for m in all_movies if m not in watched]

    # CF scores (normalize 0 to 1)
    cf_scores = {m: model.predict(user_id, m).est for m in not_watched}
    cf_min, cf_max = min(cf_scores.values()), max(cf_scores.values())
    cf_norm = {m: (s - cf_min) / (cf_max - cf_min + 1e-9) for m, s in cf_scores.items()}

    # CB scores (normalize 0 to 1)
    cb_scores = {}
    if title in indices:
        idx = indices[title]
        for i, score in enumerate(cosine_sim[idx]):
            movie_id = movies_content.iloc[i]['movieId']
            if movie_id in cf_norm:
                cb_scores[movie_id] = score

    cb_vals = list(cb_scores.values()) if cb_scores else [0]
    cb_min, cb_max = min(cb_vals), max(cb_vals)
    cb_norm = {m: (s - cb_min) / (cb_max - cb_min + 1e-9) for m, s in cb_scores.items()}

    # combine scores
    hybrid_scores = {
        m: cf_weight * cf_norm[m] + cb_weight * cb_norm.get(m, 0)
        for m in cf_norm
    }

    top = sorted(hybrid_scores.items(), key=lambda x: x[1], reverse=True)[:n]
    result = []
    for movie_id, score in top:
        info = movies[movies['movieId'] == movie_id]
        if len(info) == 0:
            continue
        result.append({
            'title':         info['title'].values[0],
            'genres':        info['genres'].values[0],
            'hybrid_score':  round(score, 4)
        })

    return pd.DataFrame(result)

# test
print(recommend_hybrid(user_id=1, title='Toy Story (1995)', n=10))


                                               title  \
0                                 Toy Story 2 (1999)   
1                                Finding Nemo (2003)   
2                                       Shrek (2001)   
3                                          Up (2009)   
4                                 Toy Story 3 (2010)   
5  Laputa: Castle in the Sky (Tenkû no shiro Rapy...   
6    Grand Day Out with Wallace and Gromit, A (1989)   
7       My Neighbor Totoro (Tonari no Totoro) (1988)   
8             Wallace & Gromit: A Close Shave (1995)   
9  Kiki's Delivery Service (Majo no takkyûbin) (1...   

                                              genres  hybrid_score  
0        adventure animation children comedy fantasy        0.9079  
1                adventure animation children comedy        0.9032  
2  adventure animation children comedy fantasy ro...        0.9020  
3                 adventure animation children drama        0.9010  
4   adventure animation children comed

## Evaluation (Precision, Recall, F1)


In [52]:
def precision_recall_f1(predictions, k=10, threshold=3.5):
    df = pd.DataFrame(predictions, columns=['uid', 'iid', 'r_ui', 'est', 'details'])

    precisions, recalls = [], []

    for uid in df['uid'].unique():
        user_df = df[df['uid'] == uid].sort_values('est', ascending=False)
        top_k   = user_df.head(k)

        n_rel = (user_df['r_ui'] >= threshold).sum()
        n_hit = (top_k['r_ui']   >= threshold).sum()

        precisions.append(n_hit / k)
        recalls.append(n_hit / n_rel if n_rel > 0 else 0)

    p  = np.mean(precisions)
    r  = np.mean(recalls)
    f1 = 2 * p * r / (p + r) if (p + r) > 0 else 0

    return p, r, f1

In [53]:
p, r, f1 = precision_recall_f1(predictions)
print(f"RMSE: {rmse:.4f}")
print(f"MAE: {mae:.4f}")
print(f"Precision (Top 10): {p:.4f}")
print(f"Recall(Top 10) : {r:.4f}")
print(f"F1-Score(Top 10): {f1:.4f}")

RMSE: 0.8775
MAE: 0.6742
Precision (Top 10): 0.6287
Recall(Top 10) : 0.6564
F1-Score(Top 10): 0.6422
